In [45]:
from xbbg import blp
import pdblp

import pandas as pd
import numpy as np
from datetime import datetime, timedelta

from scipy.interpolate import interp1d

from scipy.interpolate import RegularGridInterpolator

import matplotlib.pyplot as plt

from scipy.stats import norm

In [ ]:
# Interpolating interest rate based on days until 
def riskFreeRateCurve(ccy, days_till):
    # US OIS Rates
    if ccy == 'USD':
        ois_tickers = [
            'USSO1Z BGN Curncy', 'USSO2Z BGN Curncy', 'USSO3Z BGN Curncy',
            'USSOA BGN Curncy', 'USSOB BGN Curncy', 'USSOC BGN Curncy', 
            'USSOF BGN Curncy', 'USSO1 BGN Curncy', 'USSO1F BGN Curncy', 
            'USSO2 BGN Curncy']

    # EUR OIS Rates
    if ccy == 'EUR':
        ois_tickers = [
            'EESWE1Z BGN Curncy', 'EESWE2Z BGN Curncy', 'EESWE3Z BGN Curncy', 
            'EESWEA BGN Curncy', 'EESWEB BGN Curncy', 'EESWEC BGN Curncy', 
            'EESWEF BGN Curncy', 'EESWE1 BGN Curncy', 'EESWE1F BGN Curncy', 
            'EESWE2 BGN Curncy']

    # GBP OIS Rates
    if ccy == 'GBP':
        ois_tickers = [
            'BPSWS1Z BGN Curncy', 'BPSWS2Z BGN Curncy', 'BPSWS3Z BGN Curncy', 
            'BPSWSA BGN Curncy', 'BPSWSB BGN Curncy', 'BPSWSC BGN Curncy', 
            'BPSWSF BGN Curncy', 'BPSWS1 BGN Curncy', 'BPSWS1F BGN Curncy', 
            'BPSWS2 BGN Curncy']

    # JPY OIS Rates
    if ccy == 'JPY':
        ois_tickers = [
            'JYSO1Z BGN Curncy', 'JYSO2Z BGN Curncy', 'JYSO3Z BGN Curncy', # 3W
            'JYSOA BGN Curncy', 'JYSOB BGN Curncy', 'JYSOC BGN Curncy', # 3M
            'JYSOF BGN Curncy', 'JYSO1 BGN Curncy', 'JYSO1F BGN Curncy', # 18M
            'JYSO2 BGN Curncy']


    # CAD OIS Rates
    if ccy == 'CAD':
        ois_tickers = [
            'CDSO1Z BGN Curncy', 'CDSO2Z BGN Curncy', 'CDSO3Z BGN Curncy', # 3W
            'CDSOA BGN Curncy', 'CDSOB BGN Curncy', 'CDSOC BGN Curncy', # 3M
            'CDSOF BGN Curncy', 'CDSO1 BGN Curncy', 'CDSO1F BGN Curncy', # 18M
            'CDSO2 BGN Curncy']



    # CHF OIS Rates
    if ccy == 'CHF':
        ois_tickers = [
            'SFSNT1Z BGN Curncy', 'SFSNT2Z BGN Curncy', 'SFSNT3Z BGN Curncy', # 3W
            'SFSNTA BGN Curncy', 'SFSNTB BGN Curncy', 'SFSNTC BGN Curncy', # 3M
            'SFSNTF BGN Curncy', 'SFSNT1 BGN Curncy', 'SFSNT1F BGN Curncy', # 18M
            'SFSNT2 BGN Curncy']

    # AUD OIS Rates
    if ccy == 'AUD':
        ois_tickers = [
            'ADSO1Z BGN Curncy', 'ADSO2Z BGN Curncy', 'ADSO3Z BGN Curncy', # 3W
            'ADSOA BGN Curncy', 'ADSOB BGN Curncy', 'ADSOC BGN Curncy', # 3M
            'ADSOF BGN Curncy', 'ADSO1 BGN Curncy', 'ADSO1F BGN Curncy', # 18M
            'ADSO2 BGN Curncy']

    # NZD OIS Rates
    if ccy == 'NZD':
        ois_tickers = [
            'NDSO1Z BGN Curncy', 'NDSO2Z BGN Curncy', 'NDSO3Z BGN Curncy', # 3W
            'NDSOA BGN Curncy', 'NDSOB BGN Curncy', 'NDSOC BGN Curncy', # 3M
            'NDSOF BGN Curncy', 'NDSO1 BGN Curncy', 'NDSO1F BGN Curncy', # 18M
            'NDSO2 BGN Curncy']


    end_date = datetime.now().strftime("%Y-%m-%d")
    start_date = (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d")

    names = ['1W', '2W', '3W', '1M', '2M', '3M', '6M', '1Y', '18M', '2Y']

    # Gather data
    ois_data = blp.bdh(
        tickers=ois_tickers,
        flds='PX_LAST',
        start_date=start_date,
        end_date=end_date
    )

    # Clean dataframe
    ois_rates = ois_data.iloc[-1].reset_index()
    ois_rates['level_0'] = names
    ois_rates = ois_rates.drop(columns=['level_1'])
    ois_rates.columns = ['Tenor', 'Rate']
    ois_rates['Rate'] /= 100  # Convert percentages to decimals

    # Map tenors to times (in years)
    tenor_map = {'W': 7 / 365, 'M': 30 / 365, 'Y': 1}
    ois_rates['Time'] = ois_rates['Tenor'].str.extract(r'(\d+)([WMY])') \
        .apply(lambda x: int(x[0]) * tenor_map[x[1]], axis=1)

    # Interpolate the rate curve directly
    def interpolate_rate_curve(ois_curve):
        return interp1d(
            ois_curve['Time'], ois_curve['Rate'],
            kind='cubic', fill_value='extrapolate'
        )

    # Create the interpolation function
    interp_func = interpolate_rate_curve(ois_rates)
    
    # Convert days to years
    time_in_years = days_till / 365
    
    # Get the interpolated rate
    interpolated_rate = interp_func(time_in_years)
    
    return interpolated_rate

In [ ]:
# Calculate strike price(BS): Call Delta & Implied Vol  
def calcStrike_putDelta(spot, call_delta, r_domestic, r_foreign, T, implied_vol):
    epsilon = 1e-6  
    clipped_call_delta = np.clip(call_delta, epsilon, 1 - epsilon)

    d1_inverse = norm.ppf(clipped_call_delta)  
    exponent = (r_foreign - r_domestic) * T - d1_inverse * implied_vol * np.sqrt(T)
    strike = spot * np.exp(exponent)

    return strike
    


In [ ]:
# Malz Smile Model :   Get Vol from Smile measures

ccy = 'EURUSD'
tenor = '1W'
T = 7/365


rccy1 = riskFreeRateCurve('EUR', 7) # Euro 1 week Interest Rate
rccy2 = riskFreeRateCurve('USD', 7) # US 1 week Interest Rate




con = pdblp.BCon(debug=False, port=8194, timeout=5000)
con.start()
ticker = [
    f'{ccy}V{tenor} Curncy',  # ATM Vol
    f'{ccy}25R{tenor} Curncy',  # 25 Delta Risk Reversal
    f'{ccy}25B{tenor} Curncy',  # 25 Delta Butterfly
    f'{ccy} Curncy'
]       
vol_data = con.ref(ticker, ["PX_LAST"])['value']

ATM = vol_data[0] / 100  # ATM Volatility
RR = vol_data[1]  / 100    # 25 Delta Risk Reversal
Fly = vol_data[2] / 100     # 25 Delta Butterfly
spot = vol_data[3]



putDeltaValues = np.arange(0.05, 1, 0.05)
results = pd.DataFrame({
    "Put Delta": putDeltaValues,
    "Call Delta": 1 - putDeltaValues,  # Call Delta is 1 - Put Delta
})
results["Implied Vol"] = ATM + 2 * RR * (results["Put Delta"] - 0.5) + 16 * Fly * (results["Put Delta"] - 0.5)**2
results["Implied Vol"] = results["Implied Vol"] * 100


# Strikes based on Call Deltas
results["Strike"] = results.apply(
    lambda row: calcStrike_putDelta(spot, row["Call Delta"], rccy1, rccy2, T, row["Implied Vol"] / 100), axis=1
)


In [77]:
results

,Put Delta,Call Delta,Implied Vol,Strike
0,0.05,0.95,9.012,1.013999
1,0.10,0.90,8.881,1.018844
2,0.15,0.85,8.762,1.022095
3,0.20,0.80,8.655,1.024642
4,0.25,0.75,8.560,1.026787
5,0.30,0.70,8.477,1.028678
6,0.35,0.65,8.406,1.030398
7,0.40,0.60,8.347,1.032003
8,0.45,0.55,8.300,1.033536
9,0.50,0.50,8.265,1.035030


In [ ]:
# LEFT OFF P272